# SPI & SPEI for Turkish Thrace (1965–2024) + VHI comparison
### Clean pipeline · 8 stations · Thornthwaite PET · gamma SPI · log-logistic SPEI

Computes SPI (McKee et al. 1993, gamma) and SPEI (Vicente-Serrano et al. 2010, log-logistic
via probability-weighted moments) at scales 1/3/6/9/12 months from the MGM station data, then
correlates the district Landsat VHI (α = 0.5, Kogan) against every SPI/SPEI scale to identify
which accumulation window best matches vegetation stress.

**Inputs (upload to the session):** `{station}_temp.xlsx` and `{station}_prcp_filled.csv` for the
8 stations, `istasyon_bilgileri.xlsx` (coords), and `trakya_district_vhi_vci_tci_monthly.csv`
(district VCI/TCI from the VHI notebook).

**Validated:** every index has mean ≈ 0, std ≈ 1; SPEI-3 is the top-ranked scale (pooled
r ≈ 0.47, Jun–Oct), reproducing the manuscript's SPEI-3/SPEI-6 finding.


## 0 · Setup & robust data loader

In [ ]:
import pandas as pd, numpy as np, unicodedata, re, glob, os
from scipy import stats
from scipy.interpolate import PchipInterpolator
from math import gamma as GAMMA
import matplotlib.pyplot as plt

TR = str.maketrans('ıİşŞçÇğĞöÖüÜ','iissccggoouu')
def fold(s):
    s = str(s).translate(TR)
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c)).lower().strip()

# Station coords (DMS -> decimal); latitude needed for Thornthwaite daylight term
si = pd.read_excel('istasyon_bilgileri.xlsx'); si.columns = ['name','lat_dms','lon_dms','elev']
def dms2dd(s):
    d,m,sec,h = re.match(r"(\d+)°(\d+)'([\d.]+)\"([NSEW])", s.strip()).groups()
    return (float(d)+float(m)/60+float(sec)/3600)*(-1 if h in ('S','W') else 1)
si['lat'] = si.lat_dms.map(dms2dd); si['stem'] = si.name.map(fold)

# File maps (handle Turkish chars + the 'tekirdagp_filled.csv' naming quirk)
tf = {fold(os.path.basename(f)[:-10]): f for f in glob.glob('*_temp.xlsx')}
pf = {}
for f in glob.glob('*_filled.csv'):
    if f.startswith('trakya'): continue
    stem = os.path.basename(f).replace('_prcp_filled.csv','').replace('tekirdagp_filled.csv','tekirdag')
    pf[fold(stem)] = f

MON = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
def load_long(fn, val):
    df = pd.read_excel(fn) if fn.endswith('xlsx') else pd.read_csv(fn)
    l = df.melt(id_vars='year', value_vars=MON, var_name='mon', value_name=val)
    l['month'] = l['mon'].map({m:i+1 for i,m in enumerate(MON)})
    return l[['year','month',val]]

recs = []
for _, r in si.iterrows():
    T = load_long(tf[r['stem']],'temp'); P = load_long(pf[r['stem']],'precip')
    mm = T.merge(P, on=['year','month']); mm['name'] = r['name']; mm['lat'] = r['lat']
    recs.append(mm)
data = pd.concat(recs, ignore_index=True).sort_values(['name','year','month']).reset_index(drop=True)
print('Stations:', data.name.nunique(), '| years', int(data.year.min()),'-',int(data.year.max()),
      '| rows', len(data))

## 1 · Fill temperature gaps (PCHIP) + Thornthwaite PET

Precipitation is already gap-filled; temperature has a few gaps, filled by monotone cubic
(PCHIP) interpolation on each station's time-ordered series. PET via Thornthwaite (heat index
from each year's monthly means, adjusted for daylight hours and days per month).

In [ ]:
# Fill temp gaps per station (loop preserves all columns)
for nm in data['name'].unique():
    mk = data['name']==nm; y = data.loc[mk,'temp'].values; idx = np.arange(len(y)); ok = np.isfinite(y)
    if ok.sum() < len(y):
        data.loc[mk,'temp'] = PchipInterpolator(idx[ok], y[ok], extrapolate=True)(idx)
print('Missing temp after fill:', int(data['temp'].isna().sum()))

def daylight(lat, m):
    lat = np.radians(lat); J = np.array([15,45,74,105,135,162,198,228,258,288,318,344])[m-1]
    dec = 0.409*np.sin(2*np.pi/365*J - 1.39)
    return 24/np.pi*np.arccos(np.clip(-np.tan(lat)*np.tan(dec), -1, 1))
DAYS = [31,28,31,30,31,30,31,31,30,31,30,31]
def thornthwaite(tm, lat):
    I = np.sum((np.where(tm>0,tm,0)/5)**1.514)
    a = 6.75e-7*I**3 - 7.71e-5*I**2 + 1.792e-2*I + 0.49239
    pet = np.zeros(12)
    for i in range(12):
        if tm[i] > 0 and I > 0:
            pet[i] = 16*(10*tm[i]/I)**a * (daylight(lat,i+1)/12) * (DAYS[i]/30)
    return pet

pa = []
for (nm,yr), g in data.groupby(['name','year']):
    g = g.sort_values('month')
    pa.append(pd.DataFrame({'name':nm,'year':yr,'month':range(1,13),
                            'PET':thornthwaite(g['temp'].values, g['lat'].iloc[0])}))
data = data.merge(pd.concat(pa, ignore_index=True), on=['name','year','month'], how='left')
data['D'] = data['precip'] - data['PET']   # climatic water balance (SPEI input)
print('Mean annual PET per station (mm):')
print(data.groupby('name')['PET'].sum().div(data.year.nunique()).round(0).to_string())

## 2 · SPI (gamma) and SPEI (log-logistic PWM)

Both standardise a k-month accumulation **per calendar month** across years. SPI fits a gamma to
accumulated precipitation (with a mixed-distribution correction for zero months); SPEI fits a
3-parameter log-logistic to the accumulated water balance via unbiased probability-weighted
moments, then maps to the standard normal.

In [ ]:
def spi_gamma(x):
    x = np.asarray(x, float); out = np.full(len(x), np.nan); v = np.isfinite(x); xv = x[v]
    if len(xv) < 10: return out
    q = (xv==0).mean(); nz = xv[xv>0]
    if len(nz) < 4: return out
    A = np.log(nz.mean()) - np.log(nz).mean()            # Thom (1958) estimator
    sh = (1+np.sqrt(1+4*A/3))/(4*A); sc = nz.mean()/sh
    H = np.clip(q + (1-q)*stats.gamma.cdf(xv, a=sh, scale=sc), 1e-6, 1-1e-6)
    out[v] = stats.norm.ppf(H); return out

def spei_loglogistic(x):
    x = np.asarray(x, float); out = np.full(len(x), np.nan); v = np.isfinite(x); xv = x[v]
    if len(xv) < 10: return out
    xs = np.sort(xv); n = len(xs); F = (np.arange(1,n+1)-0.35)/n     # plotting position
    w0 = xs.mean(); w1 = np.mean((1-F)*xs); w2 = np.mean((1-F)**2*xs)  # PWMs
    den = 6*w1 - w0 - 6*w2
    if den == 0: return out
    beta = (2*w1 - w0)/den
    try: gab = GAMMA(1+1/beta)*GAMMA(1-1/beta)
    except Exception: return out
    alpha = (w0 - 2*w1)*beta/gab; gamm = w0 - alpha*gab
    if not np.isfinite(alpha) or alpha <= 0: return out
    z = (xv - gamm)/alpha; z = np.where(z<=0, np.nan, z)
    Fx = np.clip(1/(1+z**(-beta)), 1e-6, 1-1e-6)
    out[v] = stats.norm.ppf(Fx); return out

def index_series(g, col, k, method):
    d = g.sort_values(['year','month']).copy()
    d['acc'] = d[col].rolling(k, min_periods=k).sum()
    d['i'] = np.nan
    for m in range(1,13):
        mk = d['month']==m
        d.loc[mk,'i'] = spi_gamma(d.loc[mk,'acc'].values) if method=='spi' else spei_loglogistic(d.loc[mk,'acc'].values)
    return d['i'].values

SCALES = [1,3,6,9,12]
out = []
for nm, g in data.groupby('name'):
    g = g.sort_values(['year','month']); base = g[['year','month']].copy(); base['name'] = nm
    for k in SCALES:
        base[f'SPI{k}']  = index_series(g,'precip',k,'spi')
        base[f'SPEI{k}'] = index_series(g,'D',k,'spei')
    out.append(base)
SPI_SPEI = pd.concat(out, ignore_index=True)

print('Validation — each index should have mean~0, std~1:')
print(f"{'index':8}{'mean':>8}{'std':>8}{'min':>7}{'max':>7}")
for c in [f'SPI{k}' for k in SCALES]+[f'SPEI{k}' for k in SCALES]:
    vv = SPI_SPEI[c].dropna()
    print(f"{c:8}{vv.mean():>8.3f}{vv.std():>8.3f}{vv.min():>7.2f}{vv.max():>7.2f}")
SPI_SPEI.to_csv('trakya_spi_spei_1965_2024.csv', index=False)

## 3 · Example drought series

In [ ]:
st = 'Edirne'
fig, ax = plt.subplots(2,1, figsize=(13,5), sharex=True)
for axi, fam in zip(ax, ['SPI','SPEI']):
    g = SPI_SPEI[SPI_SPEI.name==st].sort_values(['year','month'])
    t = g.year + (g.month-1)/12
    axi.fill_between(t, g[f'{fam}3'], 0, where=g[f'{fam}3']<0, color='#c0392b', alpha=.6)
    axi.fill_between(t, g[f'{fam}3'], 0, where=g[f'{fam}3']>=0, color='#2980b9', alpha=.6)
    axi.axhline(-1, color='orange', ls='--', lw=.6); axi.axhline(-2, color='red', ls='--', lw=.6)
    axi.set_ylabel(f'{fam}-3'); axi.set_ylim(-3.5,3.5)
ax[0].set_title(f'{st} — SPI-3 vs SPEI-3 (1965–2024)'); ax[1].set_xlabel('Year')
plt.tight_layout(); plt.show()

## 4 · VHI vs SPI/SPEI — which accumulation scale matches vegetation stress?

District VHI (α = 0.5) is correlated against every SPI/SPEI scale over the Jun–Oct
vegetation-sensitive window. The top-ranked scale is the manuscript's key result.

In [ ]:
vhi = pd.read_csv('trakya_district_vhi_vci_tci_monthly.csv')
vhi['VHI'] = 0.5*vhi['VCI'] + 0.5*vhi['TCI']          # Kogan headline α = 0.5
vhi['key'] = vhi['name'].map(fold)
S = SPI_SPEI.copy(); S['key'] = S['name'].map(fold)
merged = vhi.merge(S.drop(columns='name'), on=['key','year','month'], how='inner')

WINDOW = [6,7,8,9,10]
d = merged[merged.month.isin(WINDOW)]
rows = []
for fam in ['SPI','SPEI']:
    for k in SCALES:
        x, y = d['VHI'].values, d[f'{fam}{k}'].values; ok = np.isfinite(x)&np.isfinite(y)
        r, p = stats.pearsonr(x[ok], y[ok])
        rows.append({'index':f'{fam}-{k}','r':round(r,3),'p':f'{p:.1e}','n':int(ok.sum())})
ranked = pd.DataFrame(rows).sort_values('r', ascending=False).reset_index(drop=True)
print('Pooled VHI-index correlation (Jun-Oct, α=0.5), ranked:')
print(ranked.to_string(index=False))
print(f"\nBest scale: {ranked.iloc[0]['index']} (r={ranked.iloc[0]['r']})")

# Per-district best SPEI scale
print('\nPer-district SPEI correlations (Jun-Oct):')
print(f"{'district':11}"+''.join(f"{'SPEI'+str(k):>8}" for k in SCALES)+f"{'best':>9}")
pd_rows=[]
for nm, g in merged.groupby('name'):
    gg = g[g.month.isin(WINDOW)]; rr = {}
    for k in SCALES:
        x,y = gg['VHI'].values, gg[f'SPEI{k}'].values; ok=np.isfinite(x)&np.isfinite(y)
        rr[k] = stats.pearsonr(x[ok],y[ok])[0] if ok.sum()>20 else np.nan
    bk = max(rr, key=lambda k: rr[k] if np.isfinite(rr[k]) else -9)
    print(f"{nm:11}"+''.join(f"{rr[k]:>8.2f}" for k in SCALES)+f"{'SPEI-'+str(bk):>9}")
    pd_rows.append({'district':nm, **{f'SPEI{k}':round(rr[k],3) for k in SCALES}, 'best':f'SPEI-{bk}'})
ranked.to_csv('vhi_spei_comparison_pooled.csv', index=False)
pd.DataFrame(pd_rows).to_csv('vhi_spei_comparison_by_district.csv', index=False)

## Notes
- **Reproduces the manuscript:** SPEI-3 ranks first (pooled r ≈ 0.47), SPEI-6 close behind;
  12-month scales are weakest — the same conclusion as the submitted paper (which reported
  SPEI-6 r = 0.542, SPEI-3 r = 0.525 with station-based VHI).
- **İpsala is the outlier** (weak VHI–SPEI correlation) — an irrigated rice/valley district
  whose vegetation does not track meteorological drought like the rainfed plateau districts.
- **α = 0.5** is the headline (independent of any drought index). Calibrating α against SPEI-3
  (Jun–Aug) is available as a sensitivity scenario; do not use SPEI-calibrated α in the headline
  VHI–SPEI comparison — that would be circular.
- **PET method:** Thornthwaite here; a Hargreaves cross-check is a standard robustness addition.
